# 1. The question and the data

*Page 1 of 9.* [index](https://robinson-vidva.github.io/mb-methylation-xai/) | [next: 2. Quality control](https://robinson-vidva.github.io/mb-methylation-xai/02_quality_control.html)

## Background

Medulloblastoma is the most common malignant brain tumour of childhood. It arises in the cerebellum.
Since 2016 the WHO classification splits it into four molecular subgroups: WNT, SHH, Group 3 and Group 4.
The subgroups differ in cell of origin, driver pathway, outcome and treatment. WNT tumours have an
excellent outcome and are candidates for therapy de-escalation. SHH tumours are driven by the
Hedgehog pathway, for which SMO inhibitors exist. Group 3 tumours often carry MYC amplification and have
the worst outcome. Group 4 is the largest subgroup and the least understood. Cavalli et al. 2017 split
the four subgroups further into twelve subtypes with distinct outcomes.

**DNA methylation.** A methyl group can be attached to a cytosine that is followed by a guanine, a CpG
site. Methylation at a gene promoter usually switches the gene off. Methylation patterns are set during
development, are stable, and survive formalin fixation. They encode the cell of origin of a tumour.
This is why methylation arrays became the clinical standard for classifying brain tumours
(Capper et al. 2018). The Illumina 450k array measures about 485,000 CpG sites. Each site gets a
**beta value**, the fraction of DNA copies that are methylated, from 0 (unmethylated) to 1 (methylated).

**Gene expression** is the amount of messenger RNA a gene produces. Here it was measured on an
Affymetrix array in the same 763 tumours. Having both measurements in the same tumours makes it possible
to test whether a methylation change actually changes a gene's output.

## Questions

1. Can a classifier trained on public beta values reproduce the published subgroup and subtype calls,
   and does it still work on tumours from another laboratory and on archival (FFPE) material?
2. Which CpG sites carry the decision, and which genes do they sit in?
3. Does promoter methylation at those sites track gene expression in the same tumours?
4. Which pathways differ between subgroups at the level of methylation and expression, and do any of
   the affected genes point to existing drugs?

Questions 1 and 2 are about diagnosis. Questions 3 and 4 are about biology and therapy. The link
between them is that a classifier is only useful beyond diagnosis if the sites it uses mean something.

## Data

All data are public and are fetched by `scripts/01_download.py`, with md5 checksums.

| Source | Content | Role here |
|---|---|---|
| GSE85212, Cavalli et al. 2017 | 763 primary medulloblastomas, 450k beta values | training and held-out test |
| GSE85217, Cavalli et al. 2017 | the same 763 tumours, Affymetrix expression | methylation versus expression |
| GSE54880, Hovestadt et al. 2013 | 276 medulloblastomas, 450k, 169 FFPE and 107 frozen | external test cohort |
| GPL13534 | Illumina 450k manifest | CpG to gene annotation |

Labels are the subgroup and subtype calls deposited with the data by the original authors.
They were themselves derived from methylation and expression clustering, so the classification part
of this project reproduces a known structure rather than discovering one.

The cohort below is split into 610 training and 153 test tumours, stratified by subtype, with a fixed
seed. The test tumours are not touched until the final evaluation.

In [1]:
import json
import pandas as pd
from IPython.display import Image, display
from pathlib import Path
TAB = Path('../results/tables')
FIG = Path('../results/figures')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 90)

In [2]:
lab = pd.read_csv('../data/processed/labels.csv', index_col=0)
pd.crosstab(lab['subtype'], lab['split'], margins=True)

split,test,train,All
subtype,,,
Group3_alpha,14,53,67
Group3_beta,7,30,37
Group3_gamma,8,32,40
Group4_alpha,20,78,98
Group4_beta,22,87,109
Group4_gamma,24,95,119
SHH_alpha,13,52,65
SHH_beta,7,28,35
SHH_delta,15,61,76


The subtypes are unbalanced. Group 4 subtypes hold around a hundred tumours each. WNT beta holds 21
in total and 4 in the test split. This matters for every subtype result that follows.

## Related work

The idea was worked out from the data and the four questions above. For completeness: a published
R package, MBMethPred (Sharif Rahmani et al. 2023, MBMethPred, Frontiers in Genetics, https://doi.org/10.3389/fgene.2023.1233657), classifies the four subgroups from the same GSE85212 cohort and
explains its models with Shapley values. It is cited here as prior work, not as a template. The
subtype task, the gene-by-gene test against expression, calibration, sparse coverage and the drug
lookup are not part of it.

*Page 1 of 9.* [index](https://robinson-vidva.github.io/mb-methylation-xai/) | [next: 2. Quality control](https://robinson-vidva.github.io/mb-methylation-xai/02_quality_control.html)